# Stage 4 — Intrinsic Karst Vulnerability (Mole Creek prototype)

Combines Stage 2 (karst susceptibility) and Stage 3 (hydrological
connectivity) into **intrinsic karst vulnerability** — still not the
final risk map. This answers: *"if something happens on the surface,
where is the underlying karst system relatively more vulnerable?"*
Land-use pressure (Stage 5) hasn't entered yet.

**Combination method:** Karst susceptibility × Hydrological connectivity
(multiplicative, not a weighted sum) — per the revised workflow plan,
matching COP's own multiplicative design (see the Framework section of
the project scope doc). Each input is normalised to 0-1 by its own
maximum before multiplying, so neither dominates just because of its
raw scale (intensity 0-4 vs. connectivity 0-3).

In [ ]:
from pathlib import Path
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

data = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = data / "Output_data"

with rasterio.open(outpath / "karst_intensity_mc_EPSG7855.tif") as src:
    intensity = src.read(1).astype(float)
    profile = src.profile
with rasterio.open(outpath / "connectivity_atlas_mc_EPSG7855.tif") as src:
    connectivity = src.read(1).astype(float)
with rasterio.open(outpath / "dem_mc_EPSG7855.tif") as src:
    dem = src.read(1)
    dem_nodata = src.nodata

valid = dem != dem_nodata

## 4.1 Normalise and combine

In [ ]:
intensity_norm = intensity / 4.0       # max KCATEGORY score from Stage 2
connectivity_norm = connectivity / 3.0  # max connectivity score from Stage 3

intrinsic = intensity_norm * connectivity_norm
intrinsic[~valid] = np.nan

print("Distinct combined score values present:", np.unique(intrinsic[valid]))

## 4.2 Classify

Only 4 distinct combined values exist (0, 0.25, 0.75, 1.0) — a direct
consequence of the discrete inputs (no "C"-category karst present at
Mole Creek, and connectivity only ever resolves to 0 or its max in this
data — see the note below). Classifying directly on these natural
values is more meaningful here than imposing arbitrary quantile breaks
on what's already a small set of discrete combinations.

In [ ]:
CLASS_LABELS = {0.0: "None", 0.25: "Low", 0.75: "High", 1.0: "Very High"}
intrinsic_class = np.full(intrinsic.shape, 255, dtype="uint8")
for i, (val, label) in enumerate(CLASS_LABELS.items()):
    intrinsic_class[valid & np.isclose(intrinsic, val)] = i

print("Class distribution (valid cells):")
for i, (val, label) in enumerate(CLASS_LABELS.items()):
    n = (intrinsic_class == i).sum()
    print(f"  {label} (score={val}): {n} cells ({n/valid.sum()*100:.1f}%)")

out_profile = profile.copy()
out_profile.update(dtype="uint8", nodata=255)
with rasterio.open(outpath / "intrinsic_vulnerability_mc_EPSG7855.tif", "w", **out_profile) as dst:
    dst.write(intrinsic_class, 1)

cmap = ListedColormap(["#f0f0f0", "#ffffb2", "#fd8d3c", "#bd0026"])
display = np.ma.masked_equal(intrinsic_class, 255)
plt.figure(figsize=(8, 8))
plt.imshow(display, cmap=cmap, vmin=0, vmax=3)
plt.colorbar(ticks=[0, 1, 2, 3], label="Intrinsic vulnerability").ax.set_yticklabels(list(CLASS_LABELS.values()))
plt.title("Mole Creek intrinsic karst vulnerability (susceptibility x connectivity)")
plt.show()

## An honest finding: connectivity barely differentiates at Mole Creek

The combined score is effectively **driven almost entirely by karst
intensity**, not connectivity — because Stage 3's connectivity layer is
close to binary in this data (85 of 90 polygons scored the maximum; see
Stage 3's own note on this). So multiplying by connectivity/3 here acts
mostly as an on/off mask (1 where any karst/catchment feature is mapped,
0 elsewhere) rather than genuinely modulating vulnerability by *how*
connected a location is.

This is worth stating plainly in `discussion.md` rather than glossed
over: at the Mole Creek scale, essentially all mapped karst is also
classed as directly connected, so this particular study area doesn't
exercise the full range of the connectivity dimension. It may behave
differently once the same method is applied **statewide** (Stage 7),
where not every mapped karst area necessarily has a proximal catchment
assigned — that's a genuine open question worth checking when we get
there, not an assumption to make now.

## Stage 4 outputs

- `intrinsic_vulnerability_mc_EPSG7855.tif` — classified intrinsic vulnerability (0=None, 1=Low, 2=High, 3=Very High)

**Next (Stage 5):** land-use pressure from DEA Level-3 land cover — the
first layer that should actually show variation *outside* the karst
polygons themselves, now that the Mole Creek boundary includes real
surrounding land (see the buffer fix in Stage 0).